In [1]:
# Install once in a notebook:
# !pip install transformers py3Dmol accelerate

import torch
import py3Dmol

from transformers import AutoTokenizer, EsmForProteinFolding
from transformers.models.esm.openfold_utils.feats import atom14_to_atom37
from transformers.models.esm.openfold_utils.protein import Protein as OFProtein
from transformers.models.esm.openfold_utils.protein import to_pdb


def outputs_to_pdb(outputs) -> list[str]:
    """Convert an EsmForProteinFolding output into PDB-formatted strings."""

    # positions contains coordinates from every recycling iteration.
    # Use the final iteration, then convert atom14 coordinates to atom37.
    final_atom_positions = atom14_to_atom37(
        outputs.positions[-1],
        outputs,
    )

    final_atom_positions = final_atom_positions.detach().cpu().numpy()

    # Move the other required tensors to NumPy.
    output_data = {
        key: value.detach().cpu().numpy()
        for key, value in outputs.items()
        if torch.is_tensor(value)
    }

    pdb_strings = []

    for batch_index in range(output_data["aatype"].shape[0]):
        protein = OFProtein(
            aatype=output_data["aatype"][batch_index],
            atom_positions=final_atom_positions[batch_index],
            atom_mask=output_data["atom37_atom_exists"][batch_index],
            residue_index=output_data["residue_index"][batch_index] + 1,
            b_factors=output_data["plddt"][batch_index],
            chain_index=(
                output_data["chain_index"][batch_index]
                if "chain_index" in output_data
                else None
            ),
        )

        pdb_strings.append(to_pdb(protein))

    return pdb_strings


# -------------------------
# Load ESMFold
# -------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "facebook/esmfold_v1"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = EsmForProteinFolding.from_pretrained(
    model_name,
    low_cpu_mem_usage=True,
).to(device)

model.eval()

sequence = "MLKNVQVQLV"

inputs = tokenizer(
    [sequence],
    return_tensors="pt",
    add_special_tokens=False,
)

inputs = {key: value.to(device) for key, value in inputs.items()}

# -------------------------
# Predict the structure
# -------------------------

with torch.no_grad():
    outputs = model(**inputs)

folded_positions = outputs.positions

# Convert prediction into PDB format.
pdb_string = outputs_to_pdb(outputs)[0]

# Save the predicted structure.
with open("predicted_structure.pdb", "w") as pdb_file:
    pdb_file.write(pdb_string)

# -------------------------
# Display the 3D structure
# -------------------------

viewer = py3Dmol.view(width=800, height=600)

viewer.addModel(pdb_string, "pdb")
viewer.setStyle(
    {
        "cartoon": {
            # PDB B-factors contain the predicted pLDDT confidence.
            "colorscheme": {
                "prop": "b",
                "gradient": "roygb",
                "min": 50,
                "max": 90,
            }
        }
    }
)

viewer.setBackgroundColor("white")
viewer.zoomTo()
viewer.show()

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/local/lib/python3.10/dist-packages/_distutils_hack/__init__.py:17: UserWarning: Distutils was imported before Setuptools, but importing Setuptools also replaces the `distutils` module in `sys.modules`. This may lead to undesirable behaviors or errors. To avoid these issues, avoid using distutils directly, ensure that setuptools is installed in the traditional way (e.g. not an editable install), and/or make sure that setuptools is always imported before distutils.
  "also replaces the `distutils` module in `sys.modules`. This may lead "
/usr/local/lib/python3.10/dist-packages/_distutils_hack/__init__.py:32: UserWarning: Setuptools is replacing distutils. Support for replacing an already imported distutils is deprecated. In the future, this c

KeyboardInterrupt: 